In [1]:
from bs4 import BeautifulSoup as bs
import pandas as pd
import numpy as np
import os
import io
import builtins
import time
import re
from itertools import product
import dash
from dash import dcc
from dash import html 
from dash import dash_table
import dash_bootstrap_components as dbc


from dash.dependencies import Input, Output
pd.set_option('display.max_rows', 150)
pd.set_option('display.max_columns', 63)

In [2]:
python_exports = "C:/Users/jackj/Documents/Sports Interactive/Football Manager 2024/Python Exports"
current_save = "Burnley"
all_players_file = "all_players_data.csv"

In [3]:
folder_path = python_exports + "/" + current_save
all_players_file_path = folder_path + "/" + all_players_file
all_folder_files = os.listdir(folder_path)


In [4]:
all_html_files = [f for f in all_folder_files if f.endswith('.html')]    
all_html_files = [w.replace('.html', '') + ' 00:00:01' for w in all_html_files]


In [5]:
if not os.path.exists(all_players_file_path):
    exported_dates = pd.DataFrame({'Date': [1, 2]})

else:
    # If the file exists, read the existing data and make a copy    
    exported_dates = pd.read_csv(all_players_file_path)

exported_dates = exported_dates.Date.unique().tolist()


In [6]:
for file in all_html_files:
    if file not in exported_dates:
        file_name = file.replace(' 00:00:01','')
        full_path = folder_path + '/' + file_name + '.html'
        fm_export= bs(open(full_path,encoding = "utf8"), "html.parser").find('table')
        raw_list = pd.read_html(io.StringIO(str(fm_export)))
        raw_table = pd.DataFrame(raw_list[0])
        date = file_name.split('.')[0] +  ' 00:00:01'
        raw_table['Date'] = pd.to_datetime(date, format = "%Y-%m-%d %H:%M:%S")
        raw_table['DoB'] = pd.to_datetime(raw_table["DoB"].str.split(" ").str[0], format="%d/%m/%Y")
        raw_table['DoB'] = raw_table['DoB'] + pd.Timedelta(seconds=1)
        raw_table['Age'] = (raw_table['Date']-raw_table['DoB']).dt.days/365.25
        raw_table['Acc'] = pd.to_numeric(raw_table['Acc'], errors='coerce')

        raw_table = raw_table.dropna(subset=['Acc'])
        if not os.path.exists(all_players_file_path):
            raw_table.to_csv(all_players_file_path, index=False)
            print(f"Created file: {all_players_file_path}")
        else:
            # If the file exists, read the existing data and make a copy
            timestr = time.strftime("%Y%m%d-%H%M%S")
            existing_df = pd.read_csv(all_players_file_path)
            csv_filename = f'C:/Users/jackj/Documents/Sports Interactive/Football Manager 2024/Python Exports/{current_save}/all_players_data_{timestr}.csv'
            existing_df.to_csv(csv_filename, index=False)

            # Union the existing DataFrame with the new DataFrame
            combined_df = pd.concat([existing_df, raw_table]).drop_duplicates().reset_index(drop=True)

            # Save the combined DataFrame back to the CSV file
            combined_df.to_csv(all_players_file_path, index=False)
            print(f"Updated file: {all_players_file_path}")


Updated file: C:/Users/jackj/Documents/Sports Interactive/Football Manager 2024/Python Exports/Burnley/all_players_data.csv


In [7]:
full_path = folder_path + '/' + 'all_players_data' + '.csv'

In [8]:
raw_table = pd.read_csv(full_path)


In [9]:
def process_segment(segment):
    segment = segment.replace("D/","Z/").replace("D ",("Z "))
    sides = re.findall(r'\((.*?)\)', segment)
    sides = sides[0]    
    lines = segment.replace("("+sides+")","").replace("/","").replace("AM","A").replace("DM","O").replace('ST','F').replace("WB","O").replace("Z","D").strip()
    result = [f"{x}{y}" for x, y in product(lines, sides)] 
        
        
    return result

In [10]:
def combine_segment(segments):
    result = []
    segments = segments.replace('GK','G (C)')
    segments = re.sub(r'\bDM\b(,?)', r'DM (C)\1', segments)
    for segment in segments.split(','):
        result.extend(process_segment(segment.strip()))
    return result

In [11]:
def apply_combinations(df, column):
    
    df["Positions"] = df[column].apply(combine_segment)
    df['Positions'] = df['Positions'].apply(replace_values)
    return df


In [12]:
def replace_values(lst):
    new_lst = []
    for item in lst:
        item = item.replace('O', 'DM')
        item = item.replace('GC', 'GK')
        item = item.replace('FC', 'ST')
        item = item.replace('A', 'AM')
        item = item.replace('DMR', 'WBR')
        item = item.replace('DML', 'WBL')
        item = item.replace('DMC', 'DM')


        new_lst.append(item)
    return new_lst

In [13]:
df = apply_combinations(raw_table, 'Position')


In [14]:
unique_values = set([item for sublist in df['Positions'] for item in sublist])


In [15]:
for position in unique_values:
    df[position] = df.apply(lambda x: 1 if position in x["Positions"] else 0, axis=1)


In [16]:
latest_export_date = df['Date'].max()
df['is_latest_export'] = df['Date'] == latest_export_date

df['GK'] = df['GK']*((5 * (df['Agi']+df['Ref']+ df['Cmd']+df['Cnt'])) + (3 * (df['1v1']+df['Ant']+df['Kic']+df['Pos'])) +(1 * (df['Acc']+df['Aer']+df['Dec']+df['Fir']+df['Han']+df['Pas']+df['TRO']+df['Vis'])))/40
df['DC'] = df['DC']*(((df['Acc']+ df['Pac']+df['Jum'])*5) +  (( df['Str'] + df['Cnt']+ df['Hea']+df['Dec'])*3) + (df['Det']+ df['Bra'] + df['Ant']+ df['Pos']))/31
df['DR'] = df['DR']*(((df['Acc']+ df['Pac']+df['Wor'])*5) +  (( df['Str'] + df['Cnt']+ df['Sta']+df['Dri']+df['Cro'])*3) + (df['Det']+ df['Dec'] + df['Agi']+ df['Pos']+df['Bal']))/35
df['DL'] = df['DL']*(((df['Acc']+ df['Pac']+df['Wor'])*5) +  (( df['Str'] + df['Cnt']+ df['Sta']+df['Dri']+df['Cro'])*3) + (df['Det']+ df['Dec'] + df['Agi']+ df['Pos']+df['Bal']))/35
df['MR'] = df['MR']*(((df['Acc']+ df['Pac']+df['Wor'])*5) +  (( df['Str'] + df['Cnt']+ df['Sta']+df['Dri']+df['Cro'])*3) + (df['Det']+ df['Dec'] + df['Agi']+ df['OtB']+df['Bal']))/35
df['ML'] = df['ML']*(((df['Acc']+ df['Pac']+df['Wor'])*5) +  (( df['Str'] + df['Cnt']+ df['Sta']+df['Dri']+df['Cro'])*3) + (df['Det']+ df['Dec'] + df['Agi']+ df['OtB']+df['Bal']))/35
df['DM'] = df['MC']*(((df['Acc']+ df['Pac']+df['Wor'])*5) +  (( df['Str'] + df['Cnt']+ df['Sta']+df['Dri']+df['Pos'])*3) + (df['Det']+ df['Dec'] + df['Agi']+ df['OtB']+df['Bal']))/35
df['AMC'] = df['MC']*(((df['Acc']+ df['Pac']+df['Wor'])*5) +  (( df['Str'] + df['Cnt']+ df['Sta']+df['Dri']+df['Vis'])*3) + (df['Det']+ df['Dec'] + df['Agi']+ df['OtB']+df['Bal']))/35
df['ST'] = df['ST']*(((df['Acc']+ df['Pac']+df['Jum'])*5) +  ((df['Dri']+ df['Ant'] + df['Cnt']+ df['Agi']+df['Bal'])*3) + (df['Det']+ df['Wor'] + df['Sta']+ df['Nat.1']))/34


In [17]:
all_ratings_df = df

In [18]:
positions_in_current_formation = ['GK', 'DC', 'DR', 'DL', 'DM','AMC', 'MR','ML', 'ST']

In [19]:
all_ratings_df['Positions'] = all_ratings_df.apply(lambda row: [col for col in positions_in_current_formation if row[col] > 0], axis=1)
all_ratings_df['Positions'] = all_ratings_df['Positions'].apply(lambda x: ', '.join(x))
all_ratings_df['Positions_filter'] = all_ratings_df['Positions']+', All'
team_df = all_ratings_df[all_ratings_df['Club'].str.contains(current_save, case=False, na=False)]


In [20]:
columns_to_calculate = ['GK', 'DC', 'DR', 'DL', 'DM', 'MR', 'ML', 'AMC', 'ST']

# Calculate the maximum value for each column in the selected subset
max_values = all_ratings_df[columns_to_calculate].max()

# Calculate the percentage of each value relative to the column maximum
percentage_df = all_ratings_df[columns_to_calculate] / max_values * 100

# Add percentage columns to the original DataFrame with new names
for col in columns_to_calculate:
    all_ratings_df[f'{col}_%'] = percentage_df[col].round(2)
    
all_ratings_df['Best Pos'] = percentage_df.idxmax(axis=1)  # Name of the column with the highest %
all_ratings_df['Best Pos Rating'] = percentage_df.max(axis=1).round(2)  # Value of the highest %

In [21]:
viz_df = all_ratings_df[['Name','Club','Age','Positions','Nat','Transfer Value','Positions_filter','GK_%','DC_%','DR_%','DL_%','DM_%','MR_%','ML_%','AMC_%','ST_%','Best Pos','Best Pos Rating','is_latest_export']]
viz_df=viz_df.round(2)
viz_df = viz_df[viz_df['is_latest_export']==True]

In [22]:


app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# Callback function to update the datatable
@app.callback(
    Output(component_id='my_squad', component_property='data'),
    [
        Input(component_id='age_range_slider', component_property='value'),
        Input(component_id='position_filter', component_property='value'),
        Input(component_id='is_latest_export_filter', component_property='value'),
        Input(component_id='club_filter', component_property='value')
    ]
)
def update_datatable(selected_age_range, selected_positions, is_latest_export, club_filter):
    # Make a copy of the global df to avoid modifying it directly
    df_filtering = viz_df.copy()
    df_filtering['Positions_filter'] = df_filtering['Positions_filter'].str.split(', ')

    # Filter the DataFrame based on the selected positions
    if 'All' in selected_positions:
        filtered_df = df_filtering
    else:
        filtered_df = df_filtering[df_filtering['Positions_filter'].apply(
            lambda x: any(pos in x for pos in selected_positions))]

    # Convert list in 'Positions' column to comma-separated strings for display
    filtered_df['Positions_filter'] = filtered_df['Positions_filter'].apply(lambda x: ', '.join(x))

    # Filter by 'is_latest_export'
   # filtered_df = filtered_df[filtered_df['is_latest_export'].isin(is_latest_export)]

    # Filter by age range
    filtered_df = filtered_df[
        (selected_age_range[0] <= filtered_df['Age']) &
        (filtered_df['Age'] <= selected_age_range[1])
    ]

    # Filter by club name
    if club_filter:  # Only filter if club_filter is not empty
        filtered_df = filtered_df[filtered_df['Club'].str.contains(club_filter, case=False, na=False)]


    # Return the filtered DataFrame as a list of dictionaries
    return filtered_df.to_dict('records')


# Define the layout of the app
app.layout = html.Div([
    dbc.Row(html.H1("Web App", style={'text-align': 'center'})),

    dbc.Row([
        dbc.Col(html.Div([
            dcc.RangeSlider(15, 40,
                            value=[15, 40],
                            id='age_range_slider'),
        ]), width=3),

        dbc.Col(html.Div([
            dcc.Dropdown(id="position_filter",
                         options=[{'label': pos, 'value': pos} for pos in ['GK', 'DC', 'DR', 'DL', 'DM', 'AMC','MR','ML', 'ST', 'All']],
                         multi=True,
                         value=['All']),  # Default value
        ]), width=3),

        dbc.Col(html.Div([
            dcc.Dropdown(id="is_latest_export_filter",
                         options=[{'label': 'True', 'value': True}, {'label': 'False', 'value': False}],
                         multi=True,
                         value=[True]),  # Default value
        ]), width=3),

        dbc.Col(html.Div([
            dcc.Input(id="club_filter",
                      type='text',
                      placeholder='Enter a club name...',
                      debounce=True,
                      value=""),  # Default value
        ]), width=3),
    ], className='row'),

    html.Br(),
    dbc.Container(
        dbc.Row(
            dbc.Col(
                html.Div(
                    id='output_container',
                    children=[
                        dash_table.DataTable(
                            columns=[{"name": i, "id": i} for i in viz_df.columns],  # Include Positions column
                            data=viz_df.to_dict('records'),
                            page_size=15,
                            id='my_squad',
                            filter_action="native",
                            sort_action="native",
                            sort_mode="multi",
                            row_selectable="multi",
                            hidden_columns=[]  # Adjust hidden columns
                        )
                    ]
                ),
            ),
            className='row'
        ), fluid=True
    )
])




In [23]:
if __name__ == '__main__':
    app.run(debug=True,jupyter_mode="external")

Dash app running on http://127.0.0.1:8050/


In [24]:

df= viz_df[viz_df['Club']=='Nottm Forest']
formation_slots = {
    "GK": 1,
    "DR": 1,
    "DC": 2,
    "DL": 1,
    "MR": 1,
    "DM": 1,
    "AMC": 1,
    "ML": 1,
    "ST": 2,
}

# Add position-specific ratings to a single column for easier selection
position_columns = ["GK_%", "DC_%", "DR_%", "DL_%", "DM_%", "MR_%", "ML_%", "AMC_%", "ST_%"]
position_map = {col[:-2]: col for col in position_columns}

# Create a melted DataFrame to handle player-position ratings easily
ratings = df.melt(
    id_vars=["Name", "Best Pos", "Best Pos Rating", "is_latest_export"],
    value_vars=position_columns,
    var_name="Position",
    value_name="Rating",
)

# Clean Position column to match "GK", "DC", etc.
ratings["Position"] = ratings["Position"].str[:-2]

# Sort by position and rating to prioritize the best fit
ratings = ratings.sort_values(by=["Position", "Rating"], ascending=[True, False])

# Function to assign players to the teams by selecting the highest-rated available player
def assign_players_to_team(formation_slots, ratings, available_players):
    team = {pos: [] for pos in formation_slots.keys()}
    # Iterate through each position and assign the highest-rated available player
    for pos, slots in formation_slots.items():
        # Select the highest-rated players for each position
        position_ratings = ratings[ratings["Position"] == pos]
        selected_players = []
        for _, row in position_ratings.iterrows():
            if row["Name"] in available_players and len(selected_players) < slots:
                selected_players.append((row["Name"], row["Rating"]))
                available_players.remove(row["Name"])
        
        team[pos] = selected_players

    return team

# Available players list for both teams (starts as all players)
available_players_for_team1 = set(df["Name"])

# Assign players to Team 1
team1 = assign_players_to_team(formation_slots, ratings, available_players_for_team1)

# Available players list for Team 2 (only those who weren't selected for Team 1)
available_players_for_team2 = set(df["Name"]) - set(player[0] for players in team1.values() for player in players)

# Assign players to Team 2 using remaining players
team2 = assign_players_to_team(formation_slots, ratings, available_players_for_team2)

# Display Team 1 and Team 2 in the desired format
def display_team(team, team_name):
    print(f"{team_name}:")
    position_order = ["GK", "DR", "DC", "DC", "DL", "MR", "DM", "AMC", "ML", "ST", "ST"]
    
    for pos in position_order:
        if pos in team:
            for player in team[pos]:
                print(f"{pos}: {player[0]} {player[1]:.2f}")
        else:
            print(f"{pos}: None")  # If no player is assigned to the position

    print("\n")

# Display both teams
display_team(team1, "Team 1")
display_team(team2, "Team 2")

Team 1:
GK: Dean Henderson 92.07
DR: Ben Godfrey 90.50
DC: Ashley Phillips 98.30
DC: Teden Mengi 96.18
DC: Ashley Phillips 98.30
DC: Teden Mengi 96.18
DL: Mirsad Ali 93.63
MR: Harrison Reed 86.13
DM: Joe Willock 92.69
AMC: Harvey Elliott 90.86
ML: Archie Brown 90.15
ST: Nathan Lowe 94.30
ST: Omari Kellyman 91.44
ST: Nathan Lowe 94.30
ST: Omari Kellyman 91.44


Team 2:
GK: Nathan Baxter 90.56
DR: Harry Clarke 89.76
DC: Travis Akomeah 95.12
DC: Nathaniel Phillips 93.42
DC: Travis Akomeah 95.12
DC: Nathaniel Phillips 93.42
DL: Tyrick Mitchell 91.20
MR: Jacob Murphy 85.04
DM: Flynn Downes 89.40
AMC: Lewis O'Brien 86.92
ML: Ethan Nwaneri 86.17
ST: Liam Delap 89.16
ST: Elijah Adebayo 87.45
ST: Liam Delap 89.16
ST: Elijah Adebayo 87.45


